# Batch pipeline - DPWM baseline + EURO-CORDEX future (many catchments)

Same **seven-step workflow** as [`../single_basin_running/pipeline_study_basins.ipynb`](../single_basin_running/pipeline_study_basins.ipynb), adapted for the **3750-basin** batch (`basins_selection.csv`). Each step calls the same Python modules; outputs are **per-basin folders** so runs are **resumable** (`SKIP_EXISTING = True`).

**Big picture**

1. **Historical** - forcing → calibration → baseline Q → thresholds (Q20, Q50, Q80).
2. **Future** - bias-correct CORDEX → future forcing → future Q → deficits vs baseline Q80.

**How to use**

| Action | What to do |
|--------|------------|
| **Pilot** | `BASIN_FILTER = "explicit"`, `EXPLICIT_BASINS = ["SE000197"]` or `LIMIT = 10` |
| **Good calibrations only** | `BASIN_FILTER = "good_cal"` (default; `KGE_invQ_cal > 0.5`) |
| **Full selection** | `BASIN_FILTER = "selection"` (~3750 basins) |
| **Calibration already done** | `RUN_STEP2_CALIBRATION = False` |
| **Run order** | CONFIG → SETUP → Status → Step 1 → … → Step 7 |

**Paths (batch layout)**

| Step | Output root |
|------|-------------|
| 1 | `Results/forcing_per_basin/{basin}/` |
| 2 | `Results/calibration_batch/params/{basin}.json` |
| 3 | `Results/discharge_components_batch/{basin}/Q_total.csv` |
| 4 | `Results/thresholds_batch/{basin}/` |
| 5 | `Results/eurocordex_bias_corrected_batch/` |
| 6 | `Results/forcing_future_per_basin/`, `discharge_components_future_batch/` |
| 7 | `Results/deficit_q80_batch/{basin}/` |

**External data:** eStreams meteo on `METEO_DIR`, streamflow CSV, `F:\Data\EUROCORDEX_MPI` (or `CORDEX_BASE`) for Steps 5–7.

In [10]:
# --- CONFIG ---
from pathlib import Path

# Which basins to process
BASIN_FILTER = "good_cal"   # "good_cal" | "selection" | "explicit"
EXPLICIT_BASINS: list[str] = []   # used when BASIN_FILTER == "explicit"
MIN_KGE_INVQ = 0.5          # for BASIN_FILTER == "good_cal"
LIMIT = 10 #None                # pilot: e.g. 10
OFFSET = 0

METEO_DIR = Path(r"F:\Data\estreams_2025\meteorology\meteorology")
CORDEX_BASE = Path(r"F:\Data\EUROCORDEX_MPI")
EXTRACTION = "areal"        # Step 5: areal | nearest

RUN_STEP1_HISTORICAL_FORCING = False
RUN_STEP2_CALIBRATION = False      # True only for new calibrations (very slow)
RUN_STEP3_BASELINE_DPWM = True
RUN_STEP4_THRESHOLDS = True
RUN_STEP5_BIAS_CORRECTION = True
RUN_STEP6_FUTURE_DPWM = True
RUN_STEP7_DEFICITS = True

SKIP_EXISTING = True
WORKERS = 8
BC_WORKERS = 1              # Step 5: CORDEX I/O heavy; often 1–4
MAX_HOURS = 1           # e.g. 9 for overnight chunk
CALIB_CHUNK_SIZE = 50       # Step 2: basins per calibrate_batch_dpwm call

THRESHOLD_YEARS = 30
SMOOTH_WINDOW_DAYS = 30
RUN_DEFICIT_PLOTS = False     # Step 7: PNG summary per basin (slow)
RUN_DEFICIT_FREQUENCY_PLOTS = False  # Step 7: annual + 2x2 seasonal CDF plots
RUN_MERGE_WIDE_THRESHOLDS = False  # optional huge wide CSVs

**Configuration — flags**

| Variable | Meaning |
|----------|---------|
| `BASIN_FILTER` | `good_cal` = params with KGE_invQ > MIN_KGE_INVQ; `selection` = all in CSV; `explicit` = EXPLICIT_BASINS |
| `RUN_STEP1` … `RUN_STEP7` | Enable/disable each step |
| `RUN_STEP2_CALIBRATION` | **Off** when `Results/calibration_batch/params/` is already populated |
| `SKIP_EXISTING` | Skip basins that already have that step's outputs |
| `WORKERS` | Parallel workers (Steps 1, 3, 4, 5, 6, 7) |
| `MAX_HOURS` | Stop starting new basins after N hours (resume next run) |

**Prerequisite chain:** 1 → 2 → 3 → 4 (baseline) and 5 → 6 → 7 (future). Step 7 needs Steps 4 and 6 for each basin.

In [ ]:
import json
import os
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
for _p in [_nb, *_nb.parents]:
    if (_p / "project_root.py").is_file() and _p.name == "notebooks":
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
from project_root import find_project_root

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from batch_pipeline_workflow import (
    DEFAULT_BC_DIR,
    DEFAULT_DEFICIT_DIR,
    DEFAULT_DISCHARGE_DIR,
    DEFAULT_DISCHARGE_FUTURE_DIR,
    DEFAULT_FORCING_DIR,
    DEFAULT_FORCING_FUTURE_DIR,
    DEFAULT_PARAMS_DIR,
    DEFAULT_THRESHOLDS_DIR,
    chunk_list,
    pipeline_status,
    resolve_basins,
)

suffix = "" if EXTRACTION == "areal" else "_nearest"
PATHS = {
    "forcing": ROOT / DEFAULT_FORCING_DIR,
    "params": ROOT / DEFAULT_PARAMS_DIR,
    "discharge": ROOT / DEFAULT_DISCHARGE_DIR,
    "thresholds": ROOT / DEFAULT_THRESHOLDS_DIR,
    "bc": ROOT / Path(str(DEFAULT_BC_DIR) + suffix) if suffix else ROOT / DEFAULT_BC_DIR,
    "forcing_future": ROOT / DEFAULT_FORCING_FUTURE_DIR,
    "discharge_future": ROOT / DEFAULT_DISCHARGE_FUTURE_DIR,
    "deficit": ROOT / DEFAULT_DEFICIT_DIR,
}

basins = resolve_basins(
    mode=BASIN_FILTER,
    explicit=EXPLICIT_BASINS,
    min_kge_invq=MIN_KGE_INVQ,
    limit=LIMIT,
    offset=OFFSET,
)
BASINS_CSV = ",".join(basins)
print("ROOT", ROOT)
print(f"Basins ({len(basins)}):", basins[:5], "..." if len(basins) > 5 else "")

## Pipeline status - what is already done?

In [13]:
status = pipeline_status(
    basins,
    forcing_dir=PATHS["forcing"],
    params_dir=PATHS["params"],
    discharge_dir=PATHS["discharge"],
    thresholds_dir=PATHS["thresholds"],
    bc_dir=PATHS["bc"],
    forcing_future_dir=PATHS["forcing_future"],
    discharge_future_dir=PATHS["discharge_future"],
    deficit_dir=PATHS["deficit"],
)
display(status)

,step,done,pending,total
0,1 forcing,3190,0,3190
1,2 calibration,3190,0,3190
2,3 baseline Q,3190,0,3190
3,4 thresholds,3190,0,3190
4,5 bias correction,6,3184,3190
5,6 future Q,0,3190,3190
6,7 deficits,0,3190,3190


## Step 1 — Historical forcing (`rainplusmelt` + PET)

Build per-basin forcing from eStreams meteo (1976–2005). **Skips** basins with both CSVs in `Results/forcing_per_basin/{basin}/`.

**CLI:** `python prepare_batch_forcing.py --skip_existing --workers 8`

In [ ]:
if RUN_STEP1_HISTORICAL_FORCING:
    from batch_pipeline_workflow import run_step1_forcing

    m1 = run_step1_forcing(
        basins,
        meteo_dir=METEO_DIR,
        out_dir=PATHS["forcing"],
        skip_existing=SKIP_EXISTING,
        workers=WORKERS,
        max_hours=MAX_HOURS,
    )
    print(m1)
else:
    print("Skipped step 1 (RUN_STEP1_HISTORICAL_FORCING=False)")

## Step 2 — Calibrate DPWM (batch PSO)

**Very slow** (~minutes per basin). Set `RUN_STEP2_CALIBRATION = False` once `Results/calibration_batch/params/{basin}.json` exists.

Uses `calibrate_batch_dpwm.py` with `--skip_existing` per chunk.

In [ ]:
if RUN_STEP2_CALIBRATION:
    import calibrate_batch_dpwm as cal

    # Step 2 usually runs on full selection; override with BASIN_FILTER if needed
    cal_basins = basins
    if BASIN_FILTER == "good_cal":
        cal_basins = resolve_basins(mode="selection", limit=LIMIT, offset=OFFSET)
        print(f"Step 2: calibrating selection list ({len(cal_basins)} basins), not only good_cal")

    for i, chunk in enumerate(chunk_list(cal_basins, CALIB_CHUNK_SIZE), start=1):
        print(f"Calibration chunk {i}/{(len(cal_basins) + CALIB_CHUNK_SIZE - 1) // CALIB_CHUNK_SIZE} ({len(chunk)} basins)")
        sys.argv = [
            "notebook",
            "--basins", ",".join(chunk),
            "--forcing_dir", str(PATHS["forcing"]),
            "--out_dir", str(PATHS["params"].parent),
            "--workers", str(WORKERS),
        ]
        if SKIP_EXISTING:
            sys.argv.append("--skip_existing")
        if MAX_HOURS is not None:
            sys.argv.extend(["--max_hours", str(MAX_HOURS)])
        cal.main()
else:
    print("Skipped step 2 (RUN_STEP2_CALIBRATION=False — reusing existing params)")

## Step 3 — Baseline DPWM (saved parameters)

Simulate historical Q using **batch params** + Step 1 forcing. **Skips** if `Q_total.csv` exists.

**CLI:** `python run_batch_baseline_dpwm.py --min_kge_invq 0.5 --skip_existing --workers 8`

In [5]:
if RUN_STEP3_BASELINE_DPWM:
    from batch_baseline_workflow import run_batch_simulate

    m3 = run_batch_simulate(
        basins,
        forcing_dir=PATHS["forcing"],
        params_dir=PATHS["params"],
        discharge_dir=PATHS["discharge"],
        skip_existing=SKIP_EXISTING,
        workers=WORKERS,
        warm_span_days=365,
        max_hours=MAX_HOURS,
    )
    (PATHS["discharge"] / "_batch_logs" / "simulate_manifest.json").write_text(
        json.dumps(m3, indent=2), encoding="utf-8"
    )
    print(m3)
else:
    print("Skipped step 3")

{'n_requested': 3190, 'n_pending': 3188, 'n_ok': 3188, 'n_failed': 0, 'stopped_early': False, 'elapsed_s': 98.9}


## Step 4 — Seasonal thresholds (Q20, Q50, Q80)

From **last 30 years** of baseline simulated Q. **Skips** if `seasonal_thresholds_last30y.csv` exists.

**CLI:** `python compute_batch_thresholds.py --min_kge_invq 0.5 --skip_existing --workers 8`

In [6]:
if RUN_STEP4_THRESHOLDS:
    from batch_baseline_workflow import run_batch_thresholds

    m4 = run_batch_thresholds(
        basins,
        discharge_dir=PATHS["discharge"],
        thresholds_dir=PATHS["thresholds"],
        skip_existing=SKIP_EXISTING,
        workers=WORKERS,
        years=THRESHOLD_YEARS,
        smooth_window_days=SMOOTH_WINDOW_DAYS,
        max_hours=MAX_HOURS,
    )
    (PATHS["thresholds"] / "_batch_logs" / "thresholds_manifest.json").write_text(
        json.dumps(m4, indent=2), encoding="utf-8"
    )
    print(m4)
else:
    print("Skipped step 4")

{'n_requested': 3190, 'n_pending': 3188, 'n_ok': 3188, 'n_failed': 0, 'stopped_early': False, 'elapsed_s': 212.5}


In [7]:
if RUN_MERGE_WIDE_THRESHOLDS:
    from batch_baseline_workflow import merge_thresholds_wide, thresholds_done

    done = [b for b in basins if thresholds_done(b, PATHS["thresholds"])]
    merge_thresholds_wide(done, thresholds_dir=PATHS["thresholds"], years=THRESHOLD_YEARS,
                          out_dir=ROOT / "Results" / "thresholds")
    print(f"Wide Q20/Q50/Q80 tables for {len(done)} basins -> Results/thresholds/")
else:
    print("Skipped wide threshold merge")

Skipped wide threshold merge


## Step 5 — Bias correction (EURO-CORDEX)

Per-basin BC outputs under `PATHS["bc"]`. **Skips** if pr/tas/pet files exist. **Requires** `CORDEX_BASE` on disk; extremely slow for thousands of basins.

Same method as the six-basin notebook (`bias_correction.py`).

In [ ]:
if RUN_STEP5_BIAS_CORRECTION:
    from batch_pipeline_workflow import run_step5_bias_correction

    m5 = run_step5_bias_correction(
        basins,
        cordex_base=CORDEX_BASE,
        bc_dir=PATHS["bc"],
        obs_dir=METEO_DIR,
        extraction=EXTRACTION,
        skip_existing=SKIP_EXISTING,
        workers=BC_WORKERS,
        max_hours=MAX_HOURS,
    )
    print(m5)
else:
    print("Skipped step 5")

[AT000009] lat=47.2604, lon=9.5789, extraction=areal
  Loaded 20 grid weights from grid_weights_AT000009.csv (sum=1.0000)
[AT000009] pr: 6+19 NC files
[AT000009] tas: 6+19 NC files
[AT000009] tasmin: 6+19 NC files
[AT000009] tasmax: 6+19 NC files
  Temperature repair: 780 day(s) had tasmin > tasmax; PET recomputed -> pet_future_bc_AT000009.csv (0 NaN remaining)
[AT000019] lat=47.4627, lon=9.6818, extraction=areal
  Loaded 4 grid weights from grid_weights_AT000019.csv (sum=1.0000)
[AT000019] pr: 6+19 NC files
[AT000019] tas: 6+19 NC files
[AT000019] tasmin: 6+19 NC files
[AT000019] tasmax: 6+19 NC files


## Step 6 — Future forcing + DPWM

Build future `rainplusmelt` + PET from Step 5, then simulate with **same batch params**. **Skips** per-basin files if they exist.

In [14]:
if RUN_STEP6_FUTURE_DPWM:
    from batch_pipeline_workflow import run_step6_future

    m6 = run_step6_future(
        basins,
        bc_dir=PATHS["bc"],
        forcing_dir=PATHS["forcing_future"],
        params_dir=PATHS["params"],
        discharge_dir=PATHS["discharge_future"],
        skip_existing=SKIP_EXISTING,
        workers=WORKERS,
        max_hours=MAX_HOURS,
    )
    print(m6)
else:
    print("Skipped step 6")

{'step': 6, 'n_requested': 3190, 'n_with_bc': 6, 'n_pending': 6, 'n_ok': 6, 'n_failed': 0, 'stopped_early': False, 'elapsed_s': 20.9}


## Step 7 — Water deficits (future Q vs baseline Q80)

Uses per-basin future Q (Step 6) and per-basin Q80 climatology (Step 4). **Skips** if `deficits_hydro_year.csv` exists.

In [15]:
if RUN_STEP7_DEFICITS:
    from batch_pipeline_workflow import run_step7_deficits

    m7 = run_step7_deficits(
        basins,
        discharge_future_dir=PATHS["discharge_future"],
        thresholds_dir=PATHS["thresholds"],
        deficit_dir=PATHS["deficit"],
        skip_existing=SKIP_EXISTING,
        workers=WORKERS,
        max_hours=MAX_HOURS,
        threshold_years=THRESHOLD_YEARS,
        plot=RUN_DEFICIT_PLOTS,
        plot_frequency=RUN_DEFICIT_FREQUENCY_PLOTS,
    )
    print(m7)
else:
    print("Skipped step 7")

{'step': 7, 'n_requested': 3190, 'n_pending': 6, 'n_ok': 6, 'n_failed': 0, 'stopped_early': False, 'elapsed_s': 4.0}


## Summary — key output checks

In [16]:
display(pipeline_status(
    basins,
    forcing_dir=PATHS["forcing"],
    params_dir=PATHS["params"],
    discharge_dir=PATHS["discharge"],
    thresholds_dir=PATHS["thresholds"],
    bc_dir=PATHS["bc"],
    forcing_future_dir=PATHS["forcing_future"],
    discharge_future_dir=PATHS["discharge_future"],
    deficit_dir=PATHS["deficit"],
))

sample = basins[0] if basins else None
if sample:
    checks = [
        PATHS["forcing"] / sample / "rainplusmelt.csv",
        PATHS["params"] / f"{sample}.json",
        PATHS["discharge"] / sample / "Q_total.csv",
        PATHS["thresholds"] / sample / "seasonal_thresholds_last30y.csv",
    ]
    for p in checks:
        print("OK  " if p.exists() else "MISS", p)

,step,done,pending,total
0,1 forcing,3190,0,3190
1,2 calibration,3190,0,3190
2,3 baseline Q,3190,0,3190
3,4 thresholds,3190,0,3190
4,5 bias correction,6,3184,3190
5,6 future Q,6,3184,3190
6,7 deficits,6,3184,3190


OK   C:\Users\eiv001\Desktop\thesis\Cursor\Results\forcing_per_basin\AT000009\rainplusmelt.csv
OK   C:\Users\eiv001\Desktop\thesis\Cursor\Results\calibration_batch\params\AT000009.json
OK   C:\Users\eiv001\Desktop\thesis\Cursor\Results\discharge_components_batch\AT000009\Q_total.csv
OK   C:\Users\eiv001\Desktop\thesis\Cursor\Results\thresholds_batch\AT000009\seasonal_thresholds_last30y.csv
